In [7]:
import os
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from itertools import groupby

In [4]:
# Load all PDFs from data folder
all_docs = []
data_dir = "data"
files = sorted([f for f in os.listdir(data_dir) if f.endswith(".pdf")])

for file in files:
    # Extract chapter number from filename (e.g., book-chapter-00.pdf -> 00)
    match = re.search(r"book-chapter-(\d+)\.pdf", file)
    if match:
        chapter_num = match.group(1)
        loader = PyPDFLoader(os.path.join(data_dir, file))
        docs = loader.load()
        
        # Add metadata to each document (page)
        for doc in docs:
            doc.metadata.update({
                "volume": 1,
                "chapter": int(chapter_num)
            })
        all_docs.extend(docs)

# Split into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Average characters per chunk
    chunk_overlap=200,   # Overlap for context across chunks
    add_start_index=True # Add chunk index in metadata for traceability
)
chunks = text_splitter.split_documents(all_docs)

In [ ]:
print(f"Split into {len(chunks)} chunks")
print("keys:")
print(vars(chunks[0]).keys())  # First chunk
print("metadata sample (first chunk):") 
print(chunks[0].metadata)
print("metadata sample (last chunk):")
print(chunks[-1].metadata)
print("page_content sample:")
print(chunks[0].page_content[:200])

Split into 259 chunks
keys:
dict_keys(['id', 'metadata', 'page_content', 'type'])
metadata sample (first chunk):
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329105746', 'source': 'data\\book-chapter-00.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'volume': 1, 'chapter': 0, 'start_index': 0}
metadata sample (last chunk):
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329110040', 'source': 'data\\book-chapter-03.pdf', 'total_pages': 24, 'page': 23, 'page_label': '24', 'volume': 1, 'chapter': 3, 'start_index': 0}
page_content sample:
Speech and Language Processing
An Introduction to Natural Language Processing,
Computational Linguistics, and Speech Recognition
with Language Models
Third Edition draft
Daniel Jurafsky
Stanford Unive


In [6]:
def group_docs_by_metadata(documents, key="source"):
    # Sort is required for groupby to work correctly
    sorted_docs = sorted(documents, key=lambda x: x.metadata.get(key, "unknown"))
    
    grouped = {
        k: list(g) 
        for k, g in groupby(sorted_docs, key=lambda x: x.metadata.get(key, "unknown"))
    }
    return grouped

In [14]:
data = group_docs_by_metadata(chunks, "chapter")
for prop in data:
    print(len(data[prop]))

31
2
136
90
